In [0]:
%sql
-- ============================================================================
-- MONTHLY ASSET SNAPSHOT — DATABRICKS MIGRATION
-- ============================================================================
-- PURPOSE  : Creates a point-in-time snapshot of all active/delivered items
--            and attachments as of the 1st of each month.
--            Table name is generated dynamically, e.g.:
--              furlenco_analytics.materialized_tables.assets_with_customers_June1_2026
--
-- MIGRATED FROM : Redshift stored procedure assets_with_customers_snapshot_monthly()
--                 (asset_with_customers.sql — Redshift version)
--
-- SCHEDULE : Run on the 1st of each month via Databricks Workflow
--            Cron: 0 0 1 * *  (midnight UTC = 05:30 IST)
--
-- SOURCE TABLES :
--   furlenco_silver.order_management_systems_evolve.items
--   furlenco_silver.order_management_systems_evolve.attachments
--   furlenco_silver.order_management_systems_evolve.snapshotted_addresses
--   furlenco_silver.order_management_systems_evolve.plans
-- ============================================================================


-- ----------------------------------------------------------------------------
-- STEP 1: Compute dynamic table name in IST
-- Produces e.g.: assets_with_customers_June1_2026
-- MMMM = full month name without padding — no trim() needed unlike Redshift
-- ----------------------------------------------------------------------------
DECLARE OR REPLACE VARIABLE full_table_name STRING DEFAULT CONCAT(
  'furlenco_finance.monthly_snapshots.assets_with_customers_',
  date_format(from_utc_timestamp(current_timestamp(), 'Asia/Kolkata'), 'MMMM'),
  '1_',
  date_format(from_utc_timestamp(current_timestamp(), 'Asia/Kolkata'), 'yyyy')
);


-- ----------------------------------------------------------------------------
-- STEP 2: Create the snapshot table using IDENTIFIER() for dynamic naming
-- UNION ALL replaces UNION — items and attachments are distinct source tables
-- with no overlapping rows, so deduplication is unnecessary overhead.
-- ----------------------------------------------------------------------------
CREATE OR REPLACE TABLE IDENTIFIER(full_table_name) AS

  WITH base AS (
    SELECT
      i.id                                                                      AS item_id,
      i.name                                                                    AS item_name,
      i.state,
      i.order_id,
      i.bundle_id,
      i.composite_item_id,
      get_json_object(CAST(i.user_details AS STRING), '$.displayId')            AS fur_id,
      DATE(from_utc_timestamp(i.tenure_start_date, 'Asia/Kolkata'))             AS tenure_start_date,
      DATE(from_utc_timestamp(i.tenure_end_date,   'Asia/Kolkata'))             AS tenure_end_date,
      DATE(from_utc_timestamp(i.delivery_date,     'Asia/Kolkata'))             AS delivery_date,
      i.tenure_in_months,
      sa.city,
      'item'                                                                    AS item_attachment,
      i.vertical,
      i.plan_id
    FROM furlenco_silver.order_management_systems_evolve.items i
    LEFT JOIN furlenco_silver.order_management_systems_evolve.snapshotted_addresses sa
      ON i.snapshotted_delivery_address_id = sa.id
    WHERE i.state NOT IN (
        'DELIVERY_SCHEDULED', 'CANCELLED', 'PICKED_UP', 'AWAITING_STOCK',
        'OUT_FOR_DELIVERY', 'CREATED', 'DELIVERY_TO_BE_SCHEDULED')
      AND DATE(from_utc_timestamp(i.delivery_date, 'Asia/Kolkata')) <=
          CAST(DATE_TRUNC('month', from_utc_timestamp(current_timestamp(), 'Asia/Kolkata')) AS DATE)
  ),

  attachment AS (
    SELECT
      a.id                                                                      AS item_id,
      a.name                                                                    AS item_name,
      a.state,
      a.order_id,
      0                                                                         AS bundle_id,
      a.composite_item_id,
      get_json_object(CAST(a.user_details AS STRING), '$.displayId')            AS fur_id,
      DATE(from_utc_timestamp(a.tenure_start_date,  'Asia/Kolkata'))            AS tenure_start_date,
      DATE(from_utc_timestamp(a.tenure_end_date,    'Asia/Kolkata'))            AS tenure_end_date,
      DATE(from_utc_timestamp(a.fulfillment_date,   'Asia/Kolkata'))            AS delivery_date,
      a.tenure_in_months,
      sa.city,
      'Attachment'                                                              AS item_attachment,
      a.vertical,
      NULL                                                                      AS plan_id
    FROM furlenco_silver.order_management_systems_evolve.attachments a
    LEFT JOIN furlenco_silver.order_management_systems_evolve.snapshotted_addresses sa
      ON a.snapshotted_delivery_address_id = sa.id
    WHERE a.state NOT IN (
        'CANCELLED', 'PICKED_UP', 'OUT_FOR_DELIVERY', 'CREATED',
        'DELIVERY_TO_BE_SCHEDULED', 'DELIVERY_SCHEDULED')
      AND DATE(from_utc_timestamp(a.fulfillment_date, 'Asia/Kolkata')) <=
          CAST(DATE_TRUNC('month', from_utc_timestamp(current_timestamp(), 'Asia/Kolkata')) AS DATE)
  )

  SELECT a.*, p.name
  FROM (SELECT * FROM base UNION ALL SELECT * FROM attachment) a
  LEFT JOIN furlenco_silver.order_management_systems_evolve.plans p
    ON a.plan_id = p.id;


-- ----------------------------------------------------------------------------
-- STEP 3: Confirm — output is visible in the Databricks Workflow run log
-- ----------------------------------------------------------------------------
SELECT
  full_table_name AS created_table,
  current_timestamp() AS created_at;
